<a href="https://colab.research.google.com/github/SushanthMorsu/AI-Resume-Screener/blob/main/Task_1_Build_a_Robust_NLP_Preprocessing_Engine_(Advanced).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1: Conceptual Understanding (NLP Questions)

### 1. Difference between "Love" and "love" in NLP
- Without normalization, NLP may treat them as different tokens.
- Lowercasing converts both into the same token for consistency.

### 2. What happens if stopwords are not removed?
- Common words like *the, is, and* dominate counts.
- Larger feature space and more noise.
- Can reduce model performance.

### 3. Two scenarios where removing stopwords is harmful
- **Sentiment Analysis:** "not happy" loses meaning if **not** is removed.
- **Search / QA:** phrases like "to be or not to be" lose intent.

### 4. Difference between stemming and lemmatization
- **Stemming:** chops suffixes using rules (running → runn).
- **Lemmatization:** returns dictionary root (running → run).

## Task 2: Build Advanced Preprocessing Function

In [1]:
import re
import string
import pandas as pd
from collections import Counter

In [2]:
def preprocess_text(text):

    # Error handling
    if not isinstance(text, str):
        return [], ""

    if text.strip() == "":
        return [], ""

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # Remove emails
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove phone-like numbers
    text = re.sub(r'\b\d+\b', ' ', text)

    # Remove repeated characters (3+ -> 1)
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove emojis / non-ascii
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenization
    tokens = text.split()

    # Remove short tokens length <=2 except no/not
    final_tokens = []
    for word in tokens:
        if len(word) > 2 or word in ['no', 'not']:
            final_tokens.append(word)

    clean_sentence = " ".join(final_tokens)

    return final_tokens, clean_sentence

## Task 3: Stress Testing

In [3]:
sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this",
    "",                     # Empty string
    "😂😂😂😂😂",            # Only emojis
    "1234567890"            # Only numbers
]

print("\nTASK 3: STRESS TESTING\n")

all_tokens = []
clean_sentences = []

for i, sentence in enumerate(sentences, 1):
    tokens, clean = preprocess_text(sentence)

    all_tokens.extend(tokens)
    clean_sentences.append(clean)

    print(f"{i}. Original Text   : {sentence}")
    print(f"   Cleaned Tokens : {tokens}")
    print(f"   Clean Sentence : {clean}")
    print("-" * 60)



TASK 3: STRESS TESTING

1. Original Text   : Get 100% FREE access now!!!
   Cleaned Tokens : ['get', 'free', 'access', 'now']
   Clean Sentence : get free access now
------------------------------------------------------------
2. Original Text   : I absolutely looooved this product 😍😍
   Cleaned Tokens : ['absolutely', 'loved', 'this', 'product']
   Clean Sentence : absolutely loved this product
------------------------------------------------------------
3. Original Text   : Worst service ever... 0/10
   Cleaned Tokens : ['worst', 'service', 'ever']
   Clean Sentence : worst service ever
------------------------------------------------------------
4. Original Text   : Call me at 9876543210
   Cleaned Tokens : ['call']
   Clean Sentence : call
------------------------------------------------------------
5. Original Text   : This is THE best course!!!
   Cleaned Tokens : ['this', 'the', 'best', 'course']
   Clean Sentence : this the best course
-----------------------------------------

## Task 4: Token Analytics


In [4]:
print("\nTASK 4: TOKEN ANALYTICS\n")

analytics = []

for sentence in sentences:
    tokens, clean = preprocess_text(sentence)

    total = len(tokens)
    unique = len(set(tokens))

    avg_len = 0
    if total > 0:
        avg_len = round(sum(len(t) for t in tokens) / total, 2)

    analytics.append({
        "Original": sentence,
        "Tokens": total,
        "Unique Tokens": unique,
        "Avg Token Length": avg_len
    })

df = pd.DataFrame(analytics)
print(df)



TASK 4: TOKEN ANALYTICS

                                 Original  Tokens  Unique Tokens  \
0             Get 100% FREE access now!!!       4              4   
1   I absolutely looooved this product 😍😍       4              4   
2              Worst service ever... 0/10       3              3   
3                   Call me at 9876543210       1              1   
4              This is THE best course!!!       4              4   
5           Visit https://openai.com now!       2              2   
6                 Nooooo this is baaad!!!       3              3   
7                       OK OK OK I got it       1              1   
8         Win $$$ now!!! Limited offer!!!       4              4   
9                I am not happy with this       4              4   
10                                              0              0   
11                                  😂😂😂😂😂       0              0   
12                             1234567890       0              0   

    Avg Token Length 

# Analysis Answers:

1. Sentence with most noise:
   'Get 100% FREE access now!!!' (numbers + symbols + uppercase + punctuation)
2. Sentence retained most meaningful tokens:
   'I am not happy with this' -> keeps sentiment-rich tokens: not, happy, with, this

## Task 5: Frequency Analysis

In [5]:
print("\nFREQUENCY ANALYSIS\n")

counter = Counter(all_tokens)

print("Top 10 Most Frequent Words:")
print(counter.most_common(10))

print("\nTop 5 Least Frequent Words:")
least = sorted(counter.items(), key=lambda x: x[1])[:5]
print(least)


FREQUENCY ANALYSIS

Top 10 Most Frequent Words:
[('this', 4), ('now', 3), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('loved', 1), ('product', 1), ('worst', 1), ('service', 1)]

Top 5 Least Frequent Words:
[('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('loved', 1)]


## Task 6: Full Pipeline


In [6]:
def full_pipeline(text_list):

    all_tokens = []
    clean_sentences = []

    for text in text_list:
        tokens, clean = preprocess_text(text)
        all_tokens.extend(tokens)
        clean_sentences.append(clean)

    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }

print("\nFULL PIPELINE OUTPUT\n")

result = full_pipeline(sentences)
print(result)


FULL PIPELINE OUTPUT

{'tokens': ['get', 'free', 'access', 'now', 'absolutely', 'loved', 'this', 'product', 'worst', 'service', 'ever', 'call', 'this', 'the', 'best', 'course', 'visit', 'now', 'no', 'this', 'bad', 'got', 'win', 'now', 'limited', 'offer', 'not', 'happy', 'with', 'this'], 'clean_sentences': ['get free access now', 'absolutely loved this product', 'worst service ever', 'call', 'this the best course', 'visit now', 'no this bad', 'got', 'win now limited offer', 'not happy with this', '', '', '']}


## Task 7: Error Handling


In [7]:
test_cases = [    "",    "😂😂😂😂",    "123456"]
for t in test_cases:
  tokens, clean = preprocess_text(t)

  print(f"Input: {repr(t)}")
  print("Tokens:", tokens)
  print("Clean Sentence:", clean)
  print("-" * 50)

Input: ''
Tokens: []
Clean Sentence: 
--------------------------------------------------
Input: '😂😂😂😂'
Tokens: []
Clean Sentence: 
--------------------------------------------------
Input: '123456'
Tokens: []
Clean Sentence: 
--------------------------------------------------
